# Vectorization by hand

Implement Vocabulary (20 points), BagOfWordsVectorizer (30 points), and TFIDFVectorizer (50 points). Add your own code cell immediately below each implementation prompt, before its supplied demonstration and checks. Preserve all supplied cells. Use only standard-library Python for your implementations; do not use a vectorization library.

The complete NLP pipeline below is supplied, not graded, and must not be modified. It turns raw text into lowercase alphabetic strings using spaCy. No custom Token class, lemmatizer, or model download is needed. All class contracts and edge cases are listed in README.md.

In [ ]:
import math
import random
import re
from pathlib import Path

import spacy
from spacy.tokens import Doc

nlp = spacy.blank("en")


def process_review(text: str) -> list[str]:
    text = re.sub(r"<[^>]+>", " ", text)
    doc: Doc = nlp(text)
    return [token.lower_ for token in doc if token.is_alpha]


## Load the movie-review dataset

Download aclImdb_v1.tar.gz manually from https://ai.stanford.edu/~amaas/data/sentiment/ and extract it into this activity's data/ directory. You should have data/aclImdb/train/pos/, train/neg/, and test/pos/. See README.md for setup. Do not modify the supplied loader or pipeline.

The loader uses sorted filenames and random.Random(42) to sample 2,500 positive and 2,500 negative training reviews, then shuffle the combined 5,000-review subset reproducibly. Initialize this corpus once before implementing the classes. Each implementation is followed by a supplied demonstration using these reviews, then its assertions on small examples.

A separate test review is transformed with fixed vocabulary and IDF. Labels only balance the sample; classification is outside this activity. Demonstrations add no scored requirements and create individual dense vectors, not a dense matrix of all reviews.

In [ ]:
activity_root = Path.cwd()
if activity_root.name == "notebooks":
    activity_root = activity_root.parent
dataset_root = activity_root / "data" / "aclImdb"


def sample_review_paths(root: Path) -> list[Path]:
    rng = random.Random(42)
    selected = []
    for sentiment in ("pos", "neg"):
        candidates = sorted((root / "train" / sentiment).glob("*.txt"))
        if len(candidates) != 12500:
            raise FileNotFoundError(
                f"Expected 12,500 reviews in {root / 'train' / sentiment}. "
                "Download and extract the complete aclImdb v1 dataset; see README.md."
            )
        selected.extend(rng.sample(candidates, 2500))
    rng.shuffle(selected)
    return selected


review_paths = sample_review_paths(dataset_root)
review_corpus = [
    process_review(path.read_text(encoding="utf-8"))
    for path in review_paths
]
test_paths = sorted((dataset_root / "test" / "pos").glob("*.txt"))
if not test_paths:
    raise FileNotFoundError("Missing test/pos reviews; extract the complete aclImdb dataset.")
test_review = test_paths[0].read_text(encoding="utf-8")
new_document = process_review(test_review)
print(f"Training subset: {len(review_corpus)} reviews (2,500 positive, 2,500 negative)")


## First, calculate a tiny example

For the corpus below, write the vocabulary and both vectors for each document on paper. Why does `movie` have zero TF-IDF? Why is the DF of `good` one, even though it occurs twice? This is ungraded discussion.

Use first-appearance order, TF = count / document length, and IDF = `math.log(N / DF)`. No smoothing or additional normalization.

In [ ]:
toy_corpus = [
    ["good", "movie", "good"],
    ["bad", "movie"],
    ["movie"],
]


## 1. Vocabulary — 20 points

Add a code cell below defining `Vocabulary()` with `add(token: str) -> int`, `token_to_idx(token: str) -> int | None`, `idx_to_token(index: int) -> str`, and `__len__() -> int`.

Start empty; assign consecutive indexes from zero in insertion order. Adding a duplicate returns its original index. Unknown lookups return `None` without adding tokens. Negative and out-of-range indexes raise `IndexError`. Keep both lookups consistent. This class is the source of truth for vector positions.

In [ ]:
# Supplied demonstration: learn indexes from the training reviews.
review_vocabulary = Vocabulary()
for document in review_corpus:
    for token in document:
        review_vocabulary.add(token)

print("Vocabulary size:", len(review_vocabulary))
print("First 15 indexes and tokens:", [
    (index, review_vocabulary.idx_to_token(index))
    for index in range(min(15, len(review_vocabulary)))
])
sample_token = review_corpus[0][0]
sample_index = review_vocabulary.token_to_idx(sample_token)
print("Token to index:", sample_token, "->", sample_index)
print("Index to token:", sample_index, "->", review_vocabulary.idx_to_token(sample_index))


In [ ]:
# Vocabulary checks — 20 points
vocab = Vocabulary()
assert len(vocab) == 0
assert vocab.token_to_idx("missing") is None
assert vocab.add("movie") == 0
assert vocab.add("good") == 1
assert vocab.add("movie") == 0
assert vocab.add("bad") == 2
assert len(vocab) == 3
assert [vocab.idx_to_token(i) for i in range(len(vocab))] == ["movie", "good", "bad"]
assert [vocab.token_to_idx(t) for t in ["bad", "movie", "good"]] == [2, 0, 1]
assert vocab.token_to_idx("unknown") is None
assert len(vocab) == 3
for invalid_index in [-1, 3, 10]:
    try:
        vocab.idx_to_token(invalid_index)
    except IndexError:
        pass
    else:
        raise AssertionError("Invalid indexes must raise IndexError")
assert len(Vocabulary()) == 0


## 2. BagOfWordsVectorizer — 30 points

Add a code cell below defining `BagOfWordsVectorizer()` with an initially empty `.vocabulary` of type `Vocabulary`.

- `learn_corpus(corpus: list[list[str]]) -> None`: replace previous state with a vocabulary learned in document/token order.
- `vectorize(document: list[str]) -> list[int]`: return counts in vocabulary order. Use your Vocabulary for indexes; do not maintain another token-index mapping.

Ignore unknown terms without changing learned state. Empty documents give zeros. Before learning, or after learning an empty corpus, return `[]`. Use raw counts, not normalized frequencies.

In [ ]:
# Supplied demonstration: count known terms in the test review.
review_bow = BagOfWordsVectorizer()
review_bow.learn_corpus(review_corpus)
bow_vector = review_bow.vectorize(new_document)
bow_nonzero = [
    (review_bow.vocabulary.idx_to_token(index), count)
    for index, count in enumerate(bow_vector) if count != 0
]
print("Test review excerpt:", test_review[:300])
print("Vector length:", len(bow_vector))
print("First 15 nonzero counts:", bow_nonzero[:15])


In [ ]:
# BagOfWordsVectorizer checks — 30 points
bow = BagOfWordsVectorizer()
assert isinstance(bow.vocabulary, Vocabulary)
assert len(bow.vocabulary) == 0
assert bow.vectorize(["unknown"]) == []
assert bow.learn_corpus(toy_corpus) is None
assert [bow.vocabulary.idx_to_token(i) for i in range(3)] == ["good", "movie", "bad"]
for document, expected in zip(toy_corpus, [[2, 1, 0], [0, 1, 1], [0, 1, 0]]):
    result = bow.vectorize(document)
    assert isinstance(result, list)
    assert all(isinstance(value, int) for value in result)
    assert result == expected
assert bow.vectorize(["bad", "good", "bad", "unknown"]) == [1, 0, 2]
assert bow.vectorize(["unknown"]) == [0, 0, 0]
assert bow.vectorize([]) == [0, 0, 0]
assert len(bow.vocabulary) == 3
assert bow.vocabulary.token_to_idx("unknown") is None
assert bow.vectorize(toy_corpus[0]) == [2, 1, 0]
assert len(BagOfWordsVectorizer().vocabulary) == 0
bow.learn_corpus([[], ["new", "new"]])
assert len(bow.vocabulary) == 1
assert bow.vocabulary.idx_to_token(0) == "new"
assert bow.vectorize(["good", "new"]) == [1]
bow.learn_corpus([])
assert len(bow.vocabulary) == 0
assert bow.vectorize(["new"]) == []
bow.learn_corpus([[], []])
assert bow.vectorize([]) == []


## 3. TFIDFVectorizer — 50 points

Add a code cell below defining `TFIDFVectorizer()` with an initially empty `.vocabulary` of type `Vocabulary`.

- `learn_corpus(corpus: list[list[str]]) -> None`: replace all learned state, learning vocabulary and IDF values. Count each term at most once per document for DF. N includes empty documents.
- `vectorize(document: list[str]) -> list[float]`: return TF × IDF in vocabulary order. TF is the number of times the token appears in the document divided by the amount of tokens in the document; IDF is `math.log(N / DF)`.

Use Vocabulary as the source of truth for indexes. Unknown terms have no vector position and must not change vocabulary or IDF. Empty documents give zeros; an empty vocabulary gives `[]`, including before learning. Relearning an empty corpus clears state. You may reuse BagOfWordsVectorizer.

In [ ]:
# Supplied demonstration: weight terms in the same test review.
review_tfidf = TFIDFVectorizer()
review_tfidf.learn_corpus(review_corpus)
tfidf_vector = review_tfidf.vectorize(new_document)
tfidf_nonzero = [
    (review_tfidf.vocabulary.idx_to_token(index), weight)
    for index, weight in enumerate(tfidf_vector) if weight != 0
]
print("Vector length:", len(tfidf_vector))
print("First 15 nonzero TF-IDF weights:", tfidf_nonzero[:15])
print("Compare these terms with the counts in the Bag of Words demonstration.")


In [ ]:
# TFIDFVectorizer checks — 50 points
def assert_weights(actual, expected):
    assert isinstance(actual, list)
    assert len(actual) == len(expected)
    assert all(isinstance(value, float) for value in actual)
    assert all(math.isclose(a, e, rel_tol=1e-9, abs_tol=1e-12)
               for a, e in zip(actual, expected))


tfidf = TFIDFVectorizer()
assert isinstance(tfidf.vocabulary, Vocabulary)
assert len(tfidf.vocabulary) == 0
assert tfidf.vectorize(["unknown"]) == []
assert tfidf.learn_corpus(toy_corpus) is None
assert [tfidf.vocabulary.idx_to_token(i) for i in range(3)] == ["good", "movie", "bad"]
assert_weights(tfidf.vectorize(toy_corpus[0]), [2 / 3 * math.log(3), 0.0, 0.0])
assert_weights(tfidf.vectorize(toy_corpus[1]), [0.0, 0.0, math.log(3) / 2])
assert_weights(tfidf.vectorize(toy_corpus[2]), [0.0, 0.0, 0.0])
assert_weights(tfidf.vectorize(["good", "unknown"]), [math.log(3) / 2, 0.0, 0.0])
assert_weights(tfidf.vectorize(["unknown"]), [0.0, 0.0, 0.0])
assert_weights(tfidf.vectorize([]), [0.0, 0.0, 0.0])
assert len(tfidf.vocabulary) == 3
assert tfidf.vocabulary.token_to_idx("unknown") is None
assert_weights(tfidf.vectorize(toy_corpus[0]), [2 / 3 * math.log(3), 0.0, 0.0])
assert len(TFIDFVectorizer().vocabulary) == 0
tfidf.learn_corpus([["new", "new"], [], ["other", "new"]])
assert len(tfidf.vocabulary) == 2
assert [tfidf.vocabulary.idx_to_token(i) for i in range(2)] == ["new", "other"]
assert_weights(tfidf.vectorize(["new", "other"]),
               [math.log(3 / 2) / 2, math.log(3) / 2])
assert_weights(tfidf.vectorize(["good"]), [0.0, 0.0])
tfidf.learn_corpus([["new"]])
assert_weights(tfidf.vectorize(["new"]), [0.0])
tfidf.learn_corpus([])
assert len(tfidf.vocabulary) == 0
assert tfidf.vectorize(["new"]) == []
tfidf.learn_corpus([[], []])
assert tfidf.vectorize([]) == []
